Baseline Linear Regression Model – Reduced Feature Set


structured_traffic.csv

A second baseline linear regression model was developed using a reduced set of predictors in order to evaluate the predictive value of temporal traffic patterns independent of roadway-specific identifiers. This model focuses on time-based variables including hour, day of week, month, weekend indicator, and direction of travel, while excluding roadway name and segment identifier.

As with the full model, categorical variables were indexed using StringIndexer and assembled into a feature vector using VectorAssembler. This reduced model allows for direct comparison with the full-feature baseline to determine whether roadway-specific variables significantly improve predictive performance or if broader temporal patterns alone are sufficient to explain variation in traffic volume.


In [ ]:
import os

DATA_DIR = os.getenv("TRAFFIC_DATA_DIR", "../data/raw")

PREPROCESSED_CSV = f"{DATA_DIR}/01_Traffic_Volume_Counts_preprocessed.csv"
STRUCTURED_CSV = f"{DATA_DIR}/02_structured_traffic.csv"

In [0]:
df_raw = spark.read.csv(
    STRUCTURED_CSV,
    header=True,
    inferSchema=True
)

df_raw.show(5)

+----------+-------------+-----------+----------------+---------+----------+----+----------+-----------+-------+-----+----------+-------------------+--------------+
|segment_id|roadway__name|       from|              to|direction|      date|hour|hour_start|day_of_week|weekday|month|is_weekend|          timestamp|traffic_volume|
+----------+-------------+-----------+----------------+---------+----------+----+----------+-----------+-------+-----+----------+-------------------+--------------+
|     15540| BEACH STREET|UNION PLACE|VAN DUZER STREET|       NB|2012-01-09|   1|   1:00 AM|     Monday|      0|    1|     false|2012-01-09 01:00:00|          20.0|
|     15540| BEACH STREET|UNION PLACE|VAN DUZER STREET|       NB|2012-01-10|   1|   1:00 AM|    Tuesday|      1|    1|     false|2012-01-10 01:00:00|          21.0|
|     15540| BEACH STREET|UNION PLACE|VAN DUZER STREET|       NB|2012-01-11|   1|   1:00 AM|  Wednesday|      2|    1|     false|2012-01-11 01:00:00|          27.0|
|     1554

In [0]:
df_raw.printSchema()

root
 |-- segment_id: integer (nullable = true)
 |-- roadway__name: string (nullable = true)
 |-- from: string (nullable = true)
 |-- to: string (nullable = true)
 |-- direction: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- hour_start: string (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- weekday: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- traffic_volume: double (nullable = true)



In [0]:
from pyspark.sql.functions import col

df = df_raw.select(
    col("traffic_volume"),
    col("hour"),
    col("weekday"),
    col("month"),
    col("is_weekend"),
    col("direction")
)

df.show(5)

+--------------+----+-------+-----+----------+---------+
|traffic_volume|hour|weekday|month|is_weekend|direction|
+--------------+----+-------+-----+----------+---------+
|          20.0|   1|      0|    1|     false|       NB|
|          21.0|   1|      1|    1|     false|       NB|
|          27.0|   1|      2|    1|     false|       NB|
|          22.0|   1|      3|    1|     false|       NB|
|          31.0|   1|      4|    1|     false|       NB|
+--------------+----+-------+-----+----------+---------+
only showing top 5 rows


In [0]:
df = df.na.drop()

df.show(5)
print("Row count after dropping missing values:", df.count())

+--------------+----+-------+-----+----------+---------+
|traffic_volume|hour|weekday|month|is_weekend|direction|
+--------------+----+-------+-----+----------+---------+
|          20.0|   1|      0|    1|     false|       NB|
|          21.0|   1|      1|    1|     false|       NB|
|          27.0|   1|      2|    1|     false|       NB|
|          22.0|   1|      3|    1|     false|       NB|
|          31.0|   1|      4|    1|     false|       NB|
+--------------+----+-------+-----+----------+---------+
only showing top 5 rows
Row count after dropping missing values: 1023064


In [0]:
df.printSchema()

root
 |-- traffic_volume: double (nullable = true)
 |-- hour: integer (nullable = true)
 |-- weekday: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- direction: string (nullable = true)



In [0]:
from pyspark.ml.feature import StringIndexer

indexer_direction = StringIndexer(
    inputCol="direction",
    outputCol="direction_idx",
    handleInvalid="keep"
)

In [0]:
df2 = indexer_direction.fit(df).transform(df)

df2.select(
    "traffic_volume",
    "hour",
    "weekday",
    "month",
    "is_weekend",
    "direction",
    "direction_idx"
).show(5)

+--------------+----+-------+-----+----------+---------+-------------+
|traffic_volume|hour|weekday|month|is_weekend|direction|direction_idx|
+--------------+----+-------+-----+----------+---------+-------------+
|          20.0|   1|      0|    1|     false|       NB|          0.0|
|          21.0|   1|      1|    1|     false|       NB|          0.0|
|          27.0|   1|      2|    1|     false|       NB|          0.0|
|          22.0|   1|      3|    1|     false|       NB|          0.0|
|          31.0|   1|      4|    1|     false|       NB|          0.0|
+--------------+----+-------+-----+----------+---------+-------------+
only showing top 5 rows


In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "hour",
        "weekday",
        "month",
        "is_weekend",
        "direction_idx"
    ],
    outputCol="features"
)

final_data = assembler.transform(df2).select("features", "traffic_volume")
final_data.show(5, truncate=False)

+---------------------+--------------+
|features             |traffic_volume|
+---------------------+--------------+
|[1.0,0.0,1.0,0.0,0.0]|20.0          |
|[1.0,1.0,1.0,0.0,0.0]|21.0          |
|[1.0,2.0,1.0,0.0,0.0]|27.0          |
|[1.0,3.0,1.0,0.0,0.0]|22.0          |
|[1.0,4.0,1.0,0.0,0.0]|31.0          |
+---------------------+--------------+
only showing top 5 rows


In [0]:
train_data, test_data = final_data.randomSplit([0.7, 0.3], seed=42)

print("Training rows:", train_data.count())
print("Testing rows:", test_data.count())

Training rows: 716559
Testing rows: 306505


In [0]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="traffic_volume"
)

lr_model = lr.fit(train_data)

In [0]:
lr_pred = lr_model.transform(test_data)
lr_pred.select("traffic_volume", "prediction").show(10)

+--------------+------------------+
|traffic_volume|        prediction|
+--------------+------------------+
|          20.0|123.55189798055342|
|          24.0|123.55189798055342|
|          28.0|123.55189798055342|
|          34.0|123.55189798055342|
|          44.0|123.55189798055342|
|          45.0|123.55189798055342|
|          46.0|123.55189798055342|
|          57.0|123.55189798055342|
|          58.0|123.55189798055342|
|          60.0|123.55189798055342|
+--------------+------------------+
only showing top 10 rows


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse_eval = RegressionEvaluator(
    labelCol="traffic_volume",
    predictionCol="prediction",
    metricName="rmse"
)

r2_eval = RegressionEvaluator(
    labelCol="traffic_volume",
    predictionCol="prediction",
    metricName="r2"
)

mae_eval = RegressionEvaluator(
    labelCol="traffic_volume",
    predictionCol="prediction",
    metricName="mae"
)

lr_rmse = rmse_eval.evaluate(lr_pred)
lr_r2 = r2_eval.evaluate(lr_pred)
lr_mae = mae_eval.evaluate(lr_pred)

print("Baseline Linear Regression Results")
print("RMSE:", lr_rmse)
print("MAE:", lr_mae)
print("R2:", lr_r2)

Baseline Linear Regression Results
RMSE: 631.7490335108082
MAE: 360.12476963670366
R2: 0.050604041176304015


In [0]:
print("Coefficients:", lr_model.coefficients)
print("Intercept:", lr_model.intercept)

Coefficients: [18.21952490038596,0.761655089635919,21.60941561326996,-50.46845838478281,-18.593137619546965]
Intercept: 83.7229574668975


In [0]:
feature_names = ["hour", "weekday", "month", "is_weekend", "direction_idx"]

for name, coef in zip(feature_names, lr_model.coefficients):
    print(f"{name}: {coef}")

hour: 18.21952490038596
weekday: 0.761655089635919
month: 21.60941561326996
is_weekend: -50.46845838478281
direction_idx: -18.593137619546965
